## Model Monitoring Setup
Data Capture, Baseline Generation, Deployment Preparation for Model Quality Monitoring

In [16]:
# Imports & Setup
import boto3
import pandas as pd
import numpy as np
import json
import botocore
from botocore.exceptions import ClientError

import sagemaker
from sagemaker import get_execution_role, Session
from sagemaker.sklearn.model import SKLearnModel
from sagemaker.model_monitor import DataCaptureConfig, DefaultModelMonitor, CronExpressionGenerator

In [8]:
# Initialize session and role
session = sagemaker.Session()
role = get_execution_role()
region = session.boto_region_name
bucket = 'sagemaker-us-east-1-226675648827'
prefix = 'cardio_data'
s3_client = boto3.client("s3")
sm_client = boto3.client("sagemaker", region_name=region)

### Logistic Regression Endpoint

In [22]:
# Define model artifact and inference script
model_artifact = f's3://{bucket}/model/logistic/logistic_model.tar.gz'
entry_point = 'inference.py'
endpoint_name = 'cardio-logistic-monitor-endpoint'

# Create SKLearn model object
sklearn_model = SKLearnModel(
    model_data=model_artifact,
    role=role,
    entry_point=entry_point,
    framework_version='0.23-1',
    sagemaker_session=session
)

# Enable full data capture configuration
data_capture_config = DataCaptureConfig(
    enable_capture=True,
    sampling_percentage=100,
    destination_s3_uri=f's3://{bucket}/data-capture/logistic',
    capture_options=['Request', 'Response']
)

# Deploy the model if the endpoint does not already exist
def deploy_if_not_exists(model, endpoint_name, instance_type, data_capture_config):
    try:
        sm_client.describe_endpoint(EndpointName=endpoint_name)
        print(f"Endpoint '{endpoint_name}' already exists. Skipping deployment.")
    except sm_client.exceptions.ClientError as e:
        if "Could not find endpoint" in str(e):
            print(f"Deploying model to new endpoint '{endpoint_name}'...")
            model.deploy(
                initial_instance_count=1,
                instance_type=instance_type,
                endpoint_name=endpoint_name,
                data_capture_config=data_capture_config
            )
            print(f"Deployed endpoint '{endpoint_name}' successfully.")
        else:
            raise

# Call deployment function
deploy_if_not_exists(
    model=sklearn_model,
    endpoint_name=endpoint_name,
    instance_type='ml.m5.xlarge',
    data_capture_config=data_capture_config
)

Endpoint 'cardio-logistic-monitor-endpoint' already exists. Skipping deployment.


### Random Forest Endpoint

In [11]:
# Define Random Forest model artifact and endpoint name
rf_model_artifact = 's3://sagemaker-us-east-1-226675648827/model/random_forest/random_forest_model.tar.gz'
rf_entry_point = 'inference_rf.py'
rf_endpoint_name = 'cardio-rf-monitor-endpoint'

# Create the SKLearnModel object for Random Forest
rf_model = SKLearnModel(
    model_data=rf_model_artifact,
    role=role,
    entry_point=rf_entry_point,
    framework_version='0.23-1',
    sagemaker_session=session
)

# Data capture configuration for Random Forest endpoint
rf_data_capture_config = DataCaptureConfig(
    enable_capture=True,
    sampling_percentage=100,
    destination_s3_uri=f's3://{bucket}/data-capture/random-forest',
    capture_options=['Request', 'Response']
)

# Deploy if the endpoint doesn't exist
def deploy_rf_if_not_exists(model, endpoint_name, instance_type, data_capture_config):
    try:
        sm_client.describe_endpoint(EndpointName=endpoint_name)
        print(f"Endpoint '{endpoint_name}' already exists. Skipping deployment.")
    except sm_client.exceptions.ClientError as e:
        if 'Could not find endpoint' in str(e):
            print(f"Deploying Random Forest model to endpoint '{endpoint_name}'...")
            model.deploy(
                initial_instance_count=1,
                instance_type=instance_type,
                endpoint_name=endpoint_name,
                data_capture_config=data_capture_config
            )
            print(f"Deployed endpoint '{endpoint_name}' successfully.")
        else:
            raise

# Trigger deployment for Random Forest model
deploy_rf_if_not_exists(
    model=rf_model,
    endpoint_name=rf_endpoint_name,
    instance_type='ml.m5.xlarge',
    data_capture_config=rf_data_capture_config
)

Deploying Random Forest model to endpoint 'cardio-rf-monitor-endpoint'...
------!Deployed endpoint 'cardio-rf-monitor-endpoint' successfully.


<b> Check endpoint status </b>

In [8]:
# Define your endpoint names
endpoints_to_check = [
    'cardio-logistic-monitor-endpoint',
    'cardio-rf-monitor-endpoint'
]

# Initialize SageMaker client
sm_client = boto3.client('sagemaker', region_name=region)

# Check status of each endpoint
for endpoint_name in endpoints_to_check:
    try:
        response = sm_client.describe_endpoint(EndpointName=endpoint_name)
        status = response['EndpointStatus']
        print(f"Endpoint '{endpoint_name}' status: {status}")
    except sm_client.exceptions.ClientError as e:
        print(f"Endpoint '{endpoint_name}' not found or error occurred: {str(e)}")

Endpoint 'cardio-logistic-monitor-endpoint' status: InService
Endpoint 'cardio-rf-monitor-endpoint' status: InService


### Generate Baseline statistics.json and contraints.json

In [9]:
# Load the full production dataset (with target and headers)
df = pd.read_csv('s3://sagemaker-us-east-1-226675648827/cardio_data/cardio_prod_split40.csv')

# Drop the target column
df_cat = df.drop(columns=['cardio'])

# Save locally with categorical headers
cat_file = 'cardio_prod_split40_cat.csv'
df_cat.to_csv(cat_file, index=False, encoding='utf-8-sig')
print(f"Saved file as '{cat_file}'")

# Upload to S3
s3.upload_file(cat_file, bucket, f"{prefix}/{cat_file}")
print(f"Uploaded to s3://{bucket}/{prefix}/{cat_file}")

Saved file as 'cardio_prod_split40_cat.csv'
Uploaded to s3://sagemaker-us-east-1-226675648827/cardio_data/cardio_prod_split40_cat.csv


-----

### Baseline Generation Code
Using Categorial Column Data to create constraints.json and statistics.json
* <b>statistics.json: summary stats for each feature</b>
* <b>constraints.json: data quality and schema constraints</b>

In [10]:
# Initialize Model Monitor
monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    volume_size_in_gb=20,
    max_runtime_in_seconds=3600,
    sagemaker_session=session
)

# Input: Clean production data with categorical column names and no target
baseline_input = f's3://{bucket}/cardio_data/cardio_prod_split40_cat.csv'

# Output: Where statistics.json and constraints.json will be saved
baseline_output = f's3://{bucket}/monitoring/baseline-output'

# Run baseline suggestion job
monitor.suggest_baseline(
    baseline_dataset=baseline_input,
    dataset_format={'csv': {'header': True}},
    output_s3_uri=baseline_output,
    wait=True
)

print(f"Baseline generation complete. Files saved to: {baseline_output}")


Job Name:  baseline-suggestion-job-2025-06-16-21-08-42-611
Inputs:  [{'InputName': 'baseline_dataset_input', 'AppManaged': False, 'S3Input': {'S3Uri': 's3://sagemaker-us-east-1-226675648827/cardio_data/cardio_prod_split40_cat.csv', 'LocalPath': '/opt/ml/processing/input/baseline_dataset_input', 'S3DataType': 'S3Prefix', 'S3InputMode': 'File', 'S3DataDistributionType': 'FullyReplicated', 'S3CompressionType': 'None'}}]
Outputs:  [{'OutputName': 'monitoring_output', 'AppManaged': False, 'S3Output': {'S3Uri': 's3://sagemaker-us-east-1-226675648827/monitoring/baseline-output', 'LocalPath': '/opt/ml/processing/output', 'S3UploadMode': 'EndOfJob'}}]
..............2025-06-16 21:10:49.644373: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2025-06-16 21:10:49.644404: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dle

In [11]:
# Set up S3 client
baseline_prefix = 'monitoring/baseline-output/'

# List objects in the baseline output folder
response = s3.list_objects_v2(Bucket=bucket_name, Prefix=baseline_prefix)

# Extract file names
baseline_files = [obj['Key'] for obj in response.get('Contents', [])]
baseline_files

['monitoring/baseline-output/constraints.json',
 'monitoring/baseline-output/statistics.json']

------

### Create Monitors & Schedules for Each Models

In [17]:
# Baseline file locations
baseline_statistics_uri = f"s3://{bucket}/cardio_data/baseline-results/statistics.json"
baseline_constraints_uri = f"s3://{bucket}/cardio_data/baseline-results/constraints.json"

#### Logistic Regression

In [18]:
# Create monitor for Logistic Regression
log_monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=3600,
    sagemaker_session=session
)

# Create monitoring schedule
log_monitor.create_monitoring_schedule(
    monitor_schedule_name="cardio-logistic-monitor-schedule",
    endpoint_input="cardio-logistic-monitor-endpoint",
    output_s3_uri=f"s3://{bucket}/monitoring/logistic/monitor-output",
    statistics=baseline_statistics_uri,
    constraints=baseline_constraints_uri,
    enable_cloudwatch_metrics=True,
    schedule_cron_expression=CronExpressionGenerator.hourly()
)

print("Logistic Regression monitoring schedule created.")

Logistic Regression monitoring schedule created.


#### Random Forest

In [19]:
# Create monitor object for Random Forest
rf_monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=3600,
    sagemaker_session=session
)

# Create monitoring schedule for Random Forest endpoint
rf_monitor.create_monitoring_schedule(
    monitor_schedule_name="cardio-rf-monitor-schedule",
    endpoint_input="cardio-rf-monitor-endpoint",
    output_s3_uri=f"s3://{bucket}/monitoring/random-forest/monitor-output",
    statistics=baseline_statistics_uri,
    constraints=baseline_constraints_uri,
    enable_cloudwatch_metrics=True,
    schedule_cron_expression=CronExpressionGenerator.hourly()  # or .daily()
)

print("Random Forest monitoring schedule created.")

Random Forest monitoring schedule created.


In [20]:
# Verify that both monitoring schedules are active
# List of monitoring schedule names
monitoring_schedules = [
    "cardio-logistic-monitor-schedule",
    "cardio-rf-monitor-schedule"
]

# Check and print schedule statuses
for schedule in monitoring_schedules:
    response = sm_client.describe_monitoring_schedule(MonitoringScheduleName=schedule)
    status = response["MonitoringScheduleStatus"]
    last_modified = response["LastModifiedTime"]
    print(f"{schedule} — Status: {status} | Last Modified: {last_modified}")

cardio-logistic-monitor-schedule — Status: Scheduled | Last Modified: 2025-06-16 22:11:18.143000+00:00
cardio-rf-monitor-schedule — Status: Scheduled | Last Modified: 2025-06-16 22:11:36.474000+00:00


In [31]:
# Reinitialize after code execution state reset
s3 = boto3.client("s3")
bucket_name = "sagemaker-us-east-1-226675648827"

# List all objects under the data-capture/ path
response = s3.list_objects_v2(Bucket=bucket_name, Prefix="data-capture/")

# Extract folder prefixes (e.g., logistic/, random-forest/)
prefixes = set()
if "Contents" in response:
    for obj in response["Contents"]:
        parts = obj["Key"].split("/")
        if len(parts) > 1:
            prefixes.add(parts[1])

prefixes

{'logistic', 'random-forest'}

In [ ]:
!aws s3 cp cardio_monitoring_endpoint_scheduling.ipynb s3://sagemaker-us-east-1-226675648827/monitoring/cardio_monitoring_endpoint_scheduling.ipynb